# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [12]:
import re
import numpy as np
import pandas as pd

# 1. Carga del corpus Wikipedia Text Corpus for NLP and LLM Projects
df = pd.read_csv('wikipedia_text_corpus.csv')
corpus = df['text'].dropna().tolist()
doc_names = [doc.split('\n')[0][:80] for doc in corpus]
print(f"Artículos cargados: {len(corpus)}")

# 2. Preprocesamiento: minúsculas + eliminar caracteres no alfanuméricos
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

corpus_processed = [preprocess(doc) for doc in corpus]
print(f"Ejemplo procesado:\n{corpus_processed[0][:200]}")


Artículos cargados: 10859
Ejemplo procesado:
anovo anovo formerly a novo is a computer services company based in beauvais france it was founded in 1987 went public in 1999 and is currently a member of the cac small it won in the category service


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [18]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Matriz TF y TF-IDF
count_vec = CountVectorizer(stop_words='english', min_df=5, max_df=0.95)
tf_matrix = count_vec.fit_transform(corpus_processed)

tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(tf_matrix)

vocab = count_vec.get_feature_names_out()
print(f"Matriz TF-IDF: {tfidf_matrix.shape}  (docs × términos)")

# 2. Representación vectorial — muestra de 100 docs × top 100 términos
df_values = np.asarray((tf_matrix > 0).sum(axis=0)).flatten()
top100_idx = df_values.argsort()[::-1][:100]

print(f"\nTop 100 términos más frecuentes (DF):")
for i in top100_idx:
    print(f"  {vocab[i]}: {df_values[i]} docs")

sample = pd.DataFrame(
    tfidf_matrix[:100, top100_idx].toarray(),
    columns=vocab[top100_idx]
)
print(f"\nMuestra TF-IDF (100 docs × top 100 términos):")
print(sample)

Matriz TF-IDF: (10859, 32930)  (docs × términos)

Top 100 términos más frecuentes (DF):
  used: 6285 docs
  use: 4804 docs
  time: 3957 docs
  based: 3889 docs
  using: 3851 docs
  new: 3797 docs
  high: 3475 docs
  including: 3369 docs
  technology: 3260 docs
  known: 3229 docs
  systems: 3151 docs
  called: 3128 docs
  include: 3077 docs
  different: 2910 docs
  company: 2795 docs
  developed: 2752 docs
  world: 2687 docs
  number: 2684 docs
  large: 2673 docs
  development: 2666 docs
  like: 2650 docs
  small: 2576 docs
  design: 2573 docs
  work: 2564 docs
  designed: 2536 docs
  process: 2490 docs
  available: 2443 docs
  example: 2422 docs
  early: 2421 docs
  usually: 2409 docs
  states: 2408 docs
  form: 2323 docs
  years: 2320 docs
  united: 2308 docs
  similar: 2300 docs
  later: 2294 docs
  provide: 2258 docs
  research: 2256 docs
  various: 2239 docs
  common: 2189 docs
  order: 2186 docs
  type: 2165 docs
  following: 2144 docs
  control: 2142 docs
  set: 2142 docs
  low: 

In [19]:
# 3. Función de búsqueda con TF-IDF
def buscar_tfidf(query, top_n=10):
    query_tf    = count_vec.transform([query])
    query_tfidf = tfidf_transformer.transform(query_tf)
    scores      = cosine_similarity(query_tfidf, tfidf_matrix).flatten()
    ranking_idx = scores.argsort()[::-1][:top_n]
    return pd.DataFrame({
        'Ranking'  : range(1, top_n + 1),
        'Documento': [doc_names[i] for i in ranking_idx],
        'Score'    : [scores[i] for i in ranking_idx]
    })

# 4. Verificar con 10 queries
queries = [
    "Chemical Agent Resistant Coating",
    "computer programming software",
    "cars automotive industry",
    "music instruments genres",
    "biology evolution",
    "mathematics algebra equations",
    "space planets solar system",
    "Ecuador South America",
    "economics global markets",
    "Scoccer football players",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    print(buscar_tfidf(q).to_string(index=False))


Query: 'Chemical Agent Resistant Coating'
 Ranking                                  Documento    Score
       1                           Hot melt coating 0.325545
       2           Chemical Agent Resistant Coating 0.312883
       3                          Chemical engineer 0.278768
       4                           Chemical process 0.277258
       5                    Open Agent Architecture 0.239064
       6 Foundation for Intelligent Physical Agents 0.229229
       7                      Chemical technologist 0.226317
       8            Laser chemical vapor deposition 0.193316
       9                          Chemical industry 0.191741
      10                         Conversion coating 0.187839

Query: 'computer programming software'
 Ranking                 Documento    Score
       1                  Software 0.569488
       2        System programming 0.561677
       3                 Computing 0.539631
       4 Psychology of programming 0.530468
       5      Software dev

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:
# Parámetros BM25
k1 = 1.5   # controla saturación de TF
b  = 0.75  # controla normalización por longitud

# 1. Longitud de cada documento (número de términos) y promedio
doc_len = np.asarray(tf_matrix.sum(axis=1)).flatten()
avgdl   = doc_len.mean()

# 2. DF: en cuántos documentos aparece cada término
df_bm25 = np.asarray((tf_matrix > 0).sum(axis=0)).flatten()
N = tf_matrix.shape[0]

# 3. IDF de BM25
idf_bm25 = np.log((N - df_bm25 + 0.5) / (df_bm25 + 0.5) + 1)

# 4. Función de ranking BM25
def bm25_ranking(query, top_n=5):
    query_vec   = count_vec.transform([query])
    query_terms = query_vec.nonzero()[1]
    scores      = np.zeros(N)
    for term_idx in query_terms:
        tf          = np.asarray(tf_matrix[:, term_idx].todense()).flatten()
        numerador   = tf * (k1 + 1)
        denominador = tf + k1 * (1 - b + b * doc_len / avgdl)
        scores     += idf_bm25[term_idx] * (numerador / denominador)
    ranking_idx = scores.argsort()[::-1][:top_n]
    return pd.DataFrame({
        'Ranking'   : range(1, top_n + 1),
        'Documento' : [doc_names[i] for i in ranking_idx],
        'Score BM25': [scores[i] for i in ranking_idx]
    })

# Verificar con las mismas 10 queries
for q in queries:
    print(f"\nQuery: '{q}'")
    print(bm25_ranking(q).to_string(index=False))


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [ ]:
import matplotlib.pyplot as plt

# Comparar los resultados de ambos modelos para una query
query = queries[0]

resultado_tfidf = buscar_tfidf(query, top_n=10)
resultado_bm25  = bm25_ranking(query, top_n=10)

# Unión de los top documentos de ambos modelos
todos = list(dict.fromkeys(resultado_tfidf['Documento'].tolist() + resultado_bm25['Documento'].tolist()))

# Recalcular scores TF-IDF para la unión
query_tf_vec    = count_vec.transform([query])
query_tfidf_vec = tfidf_transformer.transform(query_tf_vec)
scores_tfidf    = cosine_similarity(query_tfidf_vec, tfidf_matrix).flatten()
tfidf_scores    = [scores_tfidf[doc_names.index(d)] for d in todos]

# Recalcular scores BM25 para la unión
query_terms  = query_tf_vec.nonzero()[1]
scores_bm25  = np.zeros(N)
for term_idx in query_terms:
    tf          = np.asarray(tf_matrix[:, term_idx].todense()).flatten()
    numerador   = tf * (k1 + 1)
    denominador = tf + k1 * (1 - b + b * doc_len / avgdl)
    scores_bm25 += idf_bm25[term_idx] * (numerador / denominador)
bm25_scores = [scores_bm25[doc_names.index(d)] for d in todos]

# Normalizar BM25 a [0,1] para comparar en la misma escala
bm25_scores_norm = np.array(bm25_scores) / max(bm25_scores)

# Gráfico de barras comparativo
x      = range(len(todos))
labels = [d[:30] for d in todos]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar([i - 0.2 for i in x], tfidf_scores,     width=0.4, label='TF-IDF',      color='steelblue')
ax.bar([i + 0.2 for i in x], bm25_scores_norm, width=0.4, label='BM25 (norm)', color='tomato')

ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Score (normalizado)')
ax.set_title(f"Comparación TF-IDF vs BM25 — Consulta: '{query}'")
ax.legend()
plt.tight_layout()
plt.show()
